### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    pi.category,
    pi.sub_category,
    pi.cost_price
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "ProductInfo" pi
    ON oi.product_id = pi.product_id
WHERE o.order_status IN ('Completed', 'Shipped');
"""

df_order_completed_shipped = pd.read_sql(sql, engine)

# 查看数据
df_order_completed_shipped

,order_id,customer_id,order_date,order_status,total_price_before_tax,product_id,quantity,product_name,unit_price,line_price_before_tax,category,sub_category,cost_price
0,7,93652,2023-02-26 22:38:27,Completed,118.35,169,1,Ray-Ban Clear Single vision,98.35,98.35,Lens,Clear Lens,44.55
1,13,56213,2023-03-07 09:55:45,Completed,195.56,91,1,Ray-Ban New Wayfarer Non-prescription,185.56,185.56,Sunglasses,Classic Sunglasses,97.32
2,27,72591,2023-02-24 07:01:35,Completed,225.66,28,1,Ray-Ban Kai Prescription,225.66,225.66,Sunglasses,Classic Sunglasses,113.03
3,31,37743,2023-03-06 02:52:44,Shipped,260.09,178,1,Essilor Polarized+ Progressive,317.18,260.09,Lens,Polarized+ Lens,118.27
4,34,55327,2023-01-08 23:39:39,Completed,250.84,34,1,Ray-Ban Bill Non-prescription,305.90,250.84,Eyeglasses,Eyeglasses,126.32
...,...,...,...,...,...,...,...,...,...,...,...,...,...
655896,499985,36208,2024-10-07 12:59:34,Completed,433.24,107,1,Ray-Ban Ray-Ban Reverse Prescription,433.24,433.24,AI Glasses,AI Smart Glasses,157.03
655897,499987,136134,2024-10-25 05:50:04,Completed,744.64,152,1,Ray-Ban Ray-Ban Meta Prescription,254.33,206.01,AI Glasses,AI Smart Glasses,129.20
655898,499987,136134,2024-10-25 05:50:04,Completed,744.64,40,1,Ray-Ban Ray-Ban Meta Prescription,237.84,192.65,AI Glasses,AI Smart Glasses,105.03
655899,499987,136134,2024-10-25 05:50:04,Completed,744.64,69,1,Ray-Ban Round Metal Non-prescription,427.14,345.98,Sunglasses,Classic Sunglasses,199.36


### Calculate cost per month 

In [ ]:
# 确保 order_date 是 datetime
df_order_completed_shipped["order_date"] = pd.to_datetime(df_order_completed_shipped["order_date"])

# 增加月份列
df_order_completed_shipped["order_month"] = df_order_completed_shipped["order_date"].dt.to_period("M").astype(str)


### Calculate `total_price_before_tax` per order 

In [ ]:
# Order表事先已计算好了`total_price_before_tax`了

### Calculate `gross profit` and `gross margin` per month 

In [ ]:
# 先取每张订单的收入（去重订单行）
df_revenue_per_order = (
    df_order_completed_shipped[["order_id", "order_month", "total_price_before_tax"]]
    .drop_duplicates(subset=["order_id"])
)

# 按月汇总成本、收入、毛利
df_monthly = (
    df_revenue_per_order
    .groupby("order_month", as_index=False)
    .agg(
        total_revenue=("total_price_before_tax", "sum"),
        order_count=("order_id", "nunique")
    )
)

df_monthly["avg_order_value"] = df_monthly["total_revenue"] / df_monthly["order_count"]

df_monthly

,order_month,total_revenue,order_count,avg_order_value
0,2023-01,6148562.38,15761,390.112454
1,2023-02,5576847.30,14098,395.577195
2,2023-03,5564187.32,14095,394.763201
3,2023-04,5869319.80,14851,395.213777
4,2023-05,6318166.02,15948,396.172938
5,2023-06,5644047.44,14131,399.408919
6,2023-07,8735960.26,21555,405.286952
7,2023-08,8587044.22,21464,400.067286
8,2023-09,7989478.61,19806,403.386782
9,2023-10,12402195.52,30963,400.548898


In [ ]:
# 关闭数据库连接
engine.dispose()